# Merge S8 — Postprocessing (connected-component) + (tùy chọn) 5-fold ensemble

Hai đòn bẩy nnU-Net chuẩn mà pipeline **chưa** làm (mới train fold 0, predict thô):

1. **Postprocessing (chính, RẺ, không train lại):** giữ thành phần liên thông lớn nhất mỗi lớp → bỏ đảo nhiễu. Deep-research cho thấy Dice gần như không đổi nhưng **surface error sập** (ASSD/HD95). Đây là win gần miễn phí cho biên sụn.
2. **5-fold ensemble (tùy chọn, NẶNG):** train fold 1–4 rồi trung bình softmax 5 fold → +0.5–1.5% Dice ổn định.

**CÔ LẬP (non-destructive):**
- Model d20 chỉ **ĐỌC** (Stage predict). Không sửa checkpoint, không đụng notebook cũ.
- Prediction ghi folder **MỚI** `/content/pp_*` — không đè `/content/pred_zib_ts|imo_test|skmtea` cũ.
- Postprocessing bằng script Python (scipy) → hoàn toàn tách khỏi model.
- (Native `find_best_configuration`/5-fold chỉ **thêm** file vào results, không đè fold_0.)


In [ ]:
!pip install -q nnunetv2


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 0) Config


In [ ]:
import os
from pathlib import Path
os.environ["nnUNet_raw"]          = "/content/drive/MyDrive/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"
os.environ["nnUNet_results"]      = "/content/drive/MyDrive/nnUNet_results"
os.makedirs(os.environ["nnUNet_preprocessed"], exist_ok=True)

RAW  = Path("/content/drive/MyDrive/nnUNet_raw")
ZIB  = RAW/"Dataset001_KneeOA"        # held-out ZIB Ts (GT class 1-5)
IMO  = RAW/"Dataset012_iMorphics"     # held-out iMorph test (GT 2,4,5,6,7,8)

TR, PLANS, CHK = "nnUNetTrainer_250epochs", "nnUNetResEncUNetLPlans", "checkpoint_final.pth"
REPORT = {"zib_ts":[1,2,3,4,5], "imo_test":[2,4,5,6,7,8]}   # ZIB show ca xuong de thay HD95 sap

mp = Path(os.environ["nnUNet_results"])/"Dataset020_KneeUnion"/f"{TR}__{PLANS}__3d_fullres"/"fold_0"/CHK
print("d20 checkpoint:", "OK" if mp.exists() else "MISSING", mp)
print("ZIB Ts:", len(list((ZIB/'imagesTs').glob('*_0000.nii.gz'))), "| iMorph test:", len(list((IMO/'imagesTs').glob('*_0000.nii.gz'))))


## 1) Predict d20 (RAW) trên 2 held-out → folder MỚI
Predict lại vào `/content/pp_raw_*` (không đụng pred cũ). ~15–20 phút.


In [ ]:
!nnUNetv2_predict -i {ZIB}/imagesTs -o /content/pp_raw_zib_ts  -d 20 -c 3d_fullres -p nnUNetResEncUNetLPlans -tr nnUNetTrainer_250epochs -f 0 -chk checkpoint_final.pth
!nnUNetv2_predict -i {IMO}/imagesTs -o /content/pp_raw_imo_test -d 20 -c 3d_fullres -p nnUNetResEncUNetLPlans -tr nnUNetTrainer_250epochs -f 0 -chk checkpoint_final.pth


## 2) Postprocessing — giữ thành phần liên thông lớn nhất mỗi lớp
Mỗi compartment sụn/xương là **một khối liền** → giữ CC lớn nhất per-class, bỏ đảo nhiễu (đúng cách LM-CartSeg làm). Ghi `/content/pp_clean_*`.


In [ ]:
import SimpleITK as sitk, numpy as np
from pathlib import Path

ALL_CLASSES = list(range(1,9))

def keep_largest_cc(seg, classes):
    out = np.zeros_like(seg)
    for c in classes:
        mask = (seg == c).astype(np.uint8)
        if mask.sum() == 0:
            continue
        cc = sitk.ConnectedComponent(sitk.GetImageFromArray(mask), False)
        largest = sitk.GetArrayFromImage(sitk.RelabelComponent(cc, sortByObjectSize=True)) == 1
        out[largest] = c
    return out

def postprocess_folder(raw_dir, out_dir):
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    n = 0
    for f in sorted(Path(raw_dir).glob("*.nii.gz")):
        itk = sitk.ReadImage(str(f)); arr = sitk.GetArrayFromImage(itk).astype(np.uint8)
        cleaned = keep_largest_cc(arr, ALL_CLASSES)
        o = sitk.GetImageFromArray(cleaned); o.CopyInformation(itk)
        sitk.WriteImage(o, str(Path(out_dir)/f.name)); n += 1
    print(f"{raw_dir} -> {out_dir}: {n} ca")

postprocess_folder("/content/pp_raw_zib_ts",  "/content/pp_clean_zib_ts")
postprocess_folder("/content/pp_raw_imo_test","/content/pp_clean_imo_test")


## 3) Eval RAW vs CLEAN — Dice + ASSD + HD95 (SimpleITK, không cần surface-distance)
Kỳ vọng: **Dice ~phẳng**, **ASSD/HD95 giảm** (đặc biệt HD95 nếu có đảo nhiễu xa).


In [ ]:
import SimpleITK as sitk, numpy as np
from pathlib import Path
from collections import defaultdict

def surface_dists(gm_arr, pm_arr, spacing_xyz):
    if gm_arr.sum() == 0 or pm_arr.sum() == 0: return np.nan, np.nan
    gm = sitk.GetImageFromArray(gm_arr.astype(np.uint8)); gm.SetSpacing(spacing_xyz)
    pm = sitk.GetImageFromArray(pm_arr.astype(np.uint8)); pm.SetSpacing(spacing_xyz)
    g_cont = sitk.GetArrayFromImage(sitk.LabelContour(gm, fullyConnected=True)).astype(bool)
    p_cont = sitk.GetArrayFromImage(sitk.LabelContour(pm, fullyConnected=True)).astype(bool)
    g_dm = sitk.GetArrayFromImage(sitk.Abs(sitk.SignedMaurerDistanceMap(gm, squaredDistance=False, useImageSpacing=True)))
    p_dm = sitk.GetArrayFromImage(sitk.Abs(sitk.SignedMaurerDistanceMap(pm, squaredDistance=False, useImageSpacing=True)))
    d_p2g = g_dm[p_cont]; d_g2p = p_dm[g_cont]
    if d_p2g.size == 0 or d_g2p.size == 0: return np.nan, np.nan
    alld = np.concatenate([d_p2g, d_g2p])
    return float(alld.mean()), float(np.percentile(alld, 95))

def eval_folder(gt_dir, pred_dir, classes):
    acc = defaultdict(lambda: {"d":[],"a":[],"h":[]})
    for gf in sorted(Path(gt_dir).glob("*.nii.gz")):
        pf = Path(pred_dir)/gf.name
        if not pf.exists(): continue
        g = sitk.ReadImage(str(gf)); p = sitk.ReadImage(str(pf)); sp = g.GetSpacing()
        ga = sitk.GetArrayFromImage(g); pa = sitk.GetArrayFromImage(p)
        for c in classes:
            gm, pm = ga == c, pa == c
            if gm.sum() == 0: continue
            d = 2*(gm & pm).sum()/(gm.sum()+pm.sum()+1e-8)
            a, h = surface_dists(gm, pm, sp)
            acc[c]["d"].append(d)
            if not np.isnan(a): acc[c]["a"].append(a); acc[c]["h"].append(h)
    return acc

NM = {1:"fem_bone",2:"fem_cart",3:"tib_bone",4:"med_tib",5:"lat_tib",6:"med_men",7:"lat_men",8:"patellar"}
def mean(x): return np.mean(x) if x else float("nan")

for key, gt in [("zib_ts", ZIB/"labelsTs"), ("imo_test", IMO/"labelsTs")]:
    R = eval_folder(gt, f"/content/pp_raw_{key}",   REPORT[key])
    C = eval_folder(gt, f"/content/pp_clean_{key}", REPORT[key])
    print("\n" + "="*76 + f"\n{key}  |  RAW  vs  CLEAN (keep-largest-CC)\n" + "="*76)
    print(f"{'class':11s}{'Dice_raw':>9}{'Dice_cln':>9}{'ASSD_raw':>9}{'ASSD_cln':>9}{'HD95_raw':>9}{'HD95_cln':>9}")
    for c in REPORT[key]:
        if c not in R or not R[c]["d"]: continue
        print(f"{NM[c]:11s}{mean(R[c]['d']):9.3f}{mean(C[c]['d']):9.3f}"
              f"{mean(R[c]['a']):9.2f}{mean(C[c]['a']):9.2f}{mean(R[c]['h']):9.2f}{mean(C[c]['h']):9.2f}")
print("\n-> Ky vong: Dice_cln ~ Dice_raw; ASSD_cln/HD95_cln < raw (bien sach hon).")


## 4) (TÙY CHỌN) Cách native của nnU-Net — để nnU-Net TỰ quyết CC per-class
`find_best_configuration` phân tích validation fold 0, quyết định lớp nào nên giữ-CC-lớn-nhất, và in sẵn lệnh apply. Chỉ **thêm** `postprocessing.pkl` + `crossval_results_*` vào results (không đè checkpoint).
Nếu báo thiếu fold, đây là lý do ta dùng script tay ở mục 2 (không phụ thuộc CV đủ 5 fold).


In [ ]:
!nnUNetv2_find_best_configuration 20 -c 3d_fullres -p nnUNetResEncUNetLPlans -tr nnUNetTrainer_250epochs -f 0
# Sau khi chay, doc inference_instructions.txt roi apply len pred RAW, vd:
# !nnUNetv2_apply_postprocessing -i /content/pp_raw_zib_ts -o /content/pp_nnunet_zib_ts \
#   -pp_pkl_file /content/drive/MyDrive/nnUNet_results/Dataset020_KneeUnion/{TR}__{PLANS}__3d_fullres/crossval_results_folds_0/postprocessing.pkl \
#   -np 4 -plans_json .../plans.json -dataset_json .../dataset.json


## 5) (TÙY CHỌN, NẶNG) 5-fold ensemble — +0.5–1.5% Dice ổn định
Train fold 1–4 (mỗi fold ~vài giờ, thêm vào results, KHÔNG đè fold_0), rồi predict `-f 0 1 2 3 4` (nnU-Net tự trung bình softmax). Có thể ghép luôn với postprocessing.


In [ ]:
# Train tung fold (chay lai + --c neu Colab dut). Bo comment de chay:
# !nnUNetv2_train 20 3d_fullres 1 -p nnUNetResEncUNetLPlans -tr nnUNetTrainer_250epochs
# !nnUNetv2_train 20 3d_fullres 2 -p nnUNetResEncUNetLPlans -tr nnUNetTrainer_250epochs
# !nnUNetv2_train 20 3d_fullres 3 -p nnUNetResEncUNetLPlans -tr nnUNetTrainer_250epochs
# !nnUNetv2_train 20 3d_fullres 4 -p nnUNetResEncUNetLPlans -tr nnUNetTrainer_250epochs


In [ ]:
# Sau khi co du 5 fold -> predict ensemble vao folder MOI:
# !nnUNetv2_predict -i {ZIB}/imagesTs -o /content/pp_raw5_zib_ts  -d 20 -c 3d_fullres -p nnUNetResEncUNetLPlans -tr nnUNetTrainer_250epochs -f 0 1 2 3 4 -chk checkpoint_final.pth
# !nnUNetv2_predict -i {IMO}/imagesTs -o /content/pp_raw5_imo_test -d 20 -c 3d_fullres -p nnUNetResEncUNetLPlans -tr nnUNetTrainer_250epochs -f 0 1 2 3 4 -chk checkpoint_final.pth
# postprocess_folder(...) + eval_folder(...) nhu muc 2-3 de so 5-fold vs fold-0.


## Kết luận (đọc sau khi có bảng mục 3)
- **Postprocessing:** nếu HD95/ASSD giảm rõ mà Dice không tụt → áp dụng cho mọi prediction (deliverable đẹp hơn ở biên sụn, gần như miễn phí).
- **5-fold:** nếu cần thêm Dice và có quỹ giờ → train fold 1–4 + ensemble. Đòn bẩy an toàn nhất còn lại.
- Mọi thứ ghi ID/folder mới → d20 gốc nguyên vẹn.
